In [39]:
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate,
    train_test_split,
)
from sklearn.metrics import confusion_matrix, f1_score, make_scorer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier

In [40]:
sessions = pd.read_csv("Raw_Data/challenge2_train.csv")

display(sessions.head())
sessions.info()

display(sessions["Revenue"].value_counts(dropna=False))
display(sessions["Revenue"].value_counts(normalize=True, dropna=False))

,ID,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,1,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,3,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
2,4,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
3,7,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.4,Feb,2,4,3,3,Returning_Visitor,False,False
4,9,0,0.0,0,0.0,2,37.000000,0.00,0.10,0.0,0.8,Feb,2,2,2,3,Returning_Visitor,False,False


<class 'pandas.DataFrame'>
RangeIndex: 9247 entries, 0 to 9246
Data columns (total 19 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   ID                       9247 non-null   int64  
 1   Administrative           9247 non-null   int64  
 2   Administrative_Duration  9247 non-null   float64
 3   Informational            9247 non-null   int64  
 4   Informational_Duration   9247 non-null   float64
 5   ProductRelated           9247 non-null   int64  
 6   ProductRelated_Duration  9247 non-null   float64
 7   BounceRates              9247 non-null   float64
 8   ExitRates                9247 non-null   float64
 9   PageValues               9247 non-null   float64
 10  SpecialDay               9247 non-null   float64
 11  Month                    9247 non-null   str    
 12  OperatingSystems         9247 non-null   int64  
 13  Browser                  9247 non-null   int64  
 14  Region                   9247 non-n

Revenue
False    7816
True     1431
Name: count, dtype: int64

Revenue
False    0.845247
True     0.154753
Name: proportion, dtype: float64

In [41]:
X = sessions.drop(columns=["Revenue", "ID"])
y = sessions["Revenue"].copy()

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

display(pd.DataFrame({
    "development": y_train.value_counts(normalize=True),
    "holdout": y_valid.value_counts(normalize=True),
}))

,development,holdout
Revenue,,
False,0.845208,0.845405
True,0.154792,0.154595


In [42]:
categorical = [
    "Month",
    "OperatingSystems",
    "Browser",
    "Region",
    "TrafficType",
    "VisitorType",
    "Weekend",
]

numerical = [column for column in X_train.columns if column not in categorical]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical)
    ]
)

pipelines = {
    "Logistic Regression": Pipeline([
        ("prep", preprocessor),
        ("model", LogisticRegression(max_iter=1000)),
    ]),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_purchase = make_scorer(f1_score, pos_label=True, zero_division=0)

scores = cross_validate(
    pipelines["Logistic Regression"],
    X_train,
    y_train,
    cv=cv,
    scoring=f1_purchase,
    return_train_score=True,
)

print("Mean training F1:", scores["train_score"].mean())
print("Mean CV F1:", scores["test_score"].mean())

Mean training F1: 0.5428874142703818
Mean CV F1: 0.539679676462936
